Celda 1: Definición de los Datos de Entrada (Pandas y NumPy)

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# Base de datos de materias primas e insumos para el biorreactor
datos_insumos = {
    "componente": ["Glucosa", "Extracto de Levadura", "Peptona de Carne", "Sales Fosfato"],
    "categoria": ["Carbono", "Nitrógeno", "Nitrógeno", "Minerales"],
    "costo_por_kg": [1.5, 4.8, 6.2, 2.5],       # Costos en USD para minimizar
    "disponibilidad_kg": [300, 80, 50, 150]     # Límites físicos de stock en bodega
}

df_insumos = pd.DataFrame(datos_insumos)
print("--- DATOS DE ENTRADA EN LA BODEGA ---")
df_insumos

--- DATOS DE ENTRADA EN LA BODEGA ---


,componente,categoria,costo_por_kg,disponibilidad_kg
0,Glucosa,Carbono,1.5,300
1,Extracto de Levadura,Nitrógeno,4.8,80
2,Peptona de Carne,Nitrógeno,6.2,50
3,Sales Fosfato,Minerales,2.5,150


Celda 2: Configuración Matemática y Ejecución del Optimizador

In [2]:
# 1. FUNCIÓN OBJETIVO: Coeficientes de costos (lo que queremos minimizar)
c = df_insumos["costo_por_kg"].tolist()

# 2. RESTRICCIONES DE DESIGUALDAD (A_ub * x <= b_ub)
# SciPy solo entiende "menor o igual (<=)". Como nuestros requisitos son mínimos (">="),
# multiplicamos las ecuaciones por -1 para invertir algebraicamente el signo.
A_ub = [
    [-1,  0,  0,  0],  # Restricción de Carbono mínimo: -1 * Glucosa <= -52.5
    [ 0, -1, -1,  0]   # Restricción de Nitrógeno mínimo: -1 * Extracto - 1 * Peptona <= -30.0
]
b_ub = [-52.5, -30.0]

# 3. RESTRICCIONES DE IGUALDAD (A_eq * x == b_eq)
# La suma de los 4 componentes debe ser exactamente igual a 150.0 kg
A_eq = [[1, 1, 1, 1]]
b_eq = [150.0]

# 4. LÍMITES POR VARIABLE (Bounds: 0 <= x <= stock disponible en bodega)
limites = [
    (0, df_insumos.loc[0, "disponibilidad_kg"]),  # Glucosa
    (0, df_insumos.loc[1, "disponibilidad_kg"]),  # Extracto de Levadura
    (0, df_insumos.loc[2, "disponibilidad_kg"]),  # Peptona de Carne
    (0, df_insumos.loc[3, "disponibilidad_kg"])   # Sales Fosfato
]

# 5. EJECUTAR EL OPTIMIZADOR LINEAL 
res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=limites, method='highs')

# 6. IMPRIMIR LOS RESULTADOS CON PENSAMIENTO LÓGICO
if res.success:
    print("¡OPTIMIZACIÓN EXITOSA!")
    print(f"Costo mínimo total del lote de fermentación: ${res.fun:.2f} USD\n")
    print("Desglose de la receta óptima calculada:")
    print("-" * 50)
    for i, componente in enumerate(df_insumos["componente"]):
        print(f"-> {componente}: {res.x[i]:.2f} kg (Gasto parcial: ${res.x[i] * c[i]:.2f} USD)")
else:
    print("Error: No se encontró una solución matemática que cumpla con las restricciones.")

¡OPTIMIZACIÓN EXITOSA!
Costo mínimo total del lote de fermentación: $324.00 USD

Desglose de la receta óptima calculada:
--------------------------------------------------
-> Glucosa: 120.00 kg (Gasto parcial: $180.00 USD)
-> Extracto de Levadura: 30.00 kg (Gasto parcial: $144.00 USD)
-> Peptona de Carne: 0.00 kg (Gasto parcial: $0.00 USD)
-> Sales Fosfato: 0.00 kg (Gasto parcial: $0.00 USD)


In [8]:
import sys
!{sys.executable} -m pip install pulp

  Using cached pulp-3.3.2-py3-none-any.whl.metadata (8.6 kB)
Using cached pulp-3.3.2-py3-none-any.whl (16.4 MB)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pulp

# 1. Crear el problema de optimización (Queremos MINIMIZAR el tiempo total de finalización)
prob = pulp.LpProblem("Scheduling_Biorreactor", pulp.LpMinimize)

# 2. Datos del problema: Tiempos de procesamiento (en horas) para cada lote
lotes = ["Lote_A", "Lote_B", "Lote_C"]
tiempo_proceso = {"Lote_A": 4, "Lote_B": 6, "Lote_C": 3}

# 3. VARIABLES DE DECISIÓN
# t[i]: Tiempo de inicio (continuo) de cada lote (ej: t["Lote_A"] = 0 significa que parte a la hora 0)
t = pulp.LpVariable.dicts("Tiempo_Inicio", lotes, lowBound=0, cat='Continuous')

# C_max: El tiempo final en que termina el ÚLTIMO lote de todos (lo que queremos minimizar)
C_max = pulp.LpVariable("Tiempo_Total_Operacion", lowBound=0, cat='Continuous')

# y[i, j]: Variable binaria (0 o 1) para evitar que dos lotes se crucen en el reactor.
# y[i, j] = 1 si el lote i va ANTES que el lote j. 0 de lo contrario.
pares_lotes = [(i, j) for i in lotes for j in lotes if i != j]
y = pulp.LpVariable.dicts("Precedencia", pares_lotes, cat='Binary')

# 4. FUNCIÓN OBJETIVO
# Queremos que el tiempo final de la jornada (C_max) sea el mínimo posible
prob += C_max

# 5. RESTRICCIONES
# Restricción A: El tiempo total (C_max) debe ser mayor o igual al tiempo en que termina cada lote individual.
for i in lotes:
    prob += C_max >= t[i] + tiempo_proceso[i]

# Restricción B: Lógica de exclusión mutua (Big-M). Evita que se traslapen en el biorreactor.
# Si y[i,j] = 1, entonces lote j parte después de que i termina. 
# Si y[i,j] = 0, la restricción se desactiva usando un número grande (M = 100).
M = 100  
for (i, j) in pares_lotes:
    if i < j: # Para evaluar cada par una sola vez en ambas direcciones
        # Caso 1: i va antes que j
        prob += t[j] >= t[i] + tiempo_proceso[i] - M * (1 - y[(i, j)])
        # Caso 2: j va antes que i
        prob += t[i] >= t[j] + tiempo_proceso[j] - M * y[(i, j)]

# 6. EJECUTAR EL OPTIMIZADOR
prob.solve(pulp.PULP_CBC_CMD(msg=False))

# 7. MOSTRAR LA SECUENCIA ÓPTIMA DE OPERACIONES
print(f"ESTADO DEL OPTIMIZADOR: {pulp.LpStatus[prob.status]}")
print(f"Tiempo total mínimo para liberar la planta: {pulp.value(C_max)} horas\n")
print("Cronograma (Scheduling) óptimo de carga del biorreactor:")
print("-" * 55)

# Ordenar los resultados por tiempo de inicio para ver la secuencia real
resultados = [(i, pulp.value(t[i])) for i in lotes]
resultados.sort(key=lambda x: x[1])

for lote, inicio in resultados:
    fin = inicio + tiempo_proceso[lote]
    print(f"-> {lote}: Entra a la hora {inicio:.1f} y se descarga a la hora {fin:.1f} (Duración: {tiempo_proceso[lote]}h)")

ESTADO DEL OPTIMIZADOR: Optimal
Tiempo total mínimo para liberar la planta: 13.0 horas

Cronograma (Scheduling) óptimo de carga del biorreactor:
-------------------------------------------------------
-> Lote_C: Entra a la hora 0.0 y se descarga a la hora 3.0 (Duración: 3h)
-> Lote_B: Entra a la hora 3.0 y se descarga a la hora 9.0 (Duración: 6h)
-> Lote_A: Entra a la hora 9.0 y se descarga a la hora 13.0 (Duración: 4h)


In [10]:
import sqlite3
import os

# 1. Definir la ruta del archivo de la base de datos dentro de tu carpeta 'data'
# Como estás en la carpeta 'notebooks/', usamos '../data/planta_piloto.db' para subir un nivel
db_path = os.path.join("..", "data", "planta_piloto.db")

# 2. Conectarse a la base de datos (si el archivo no existe, SQLite lo crea automáticamente)
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 3. CREAR TABLA: Usamos sintaxis SQL estándar
# Guardaremos el id del lote, la hora de inicio, fin y la duración de la operación
cursor.execute("""
CREATE TABLE IF NOT EXISTS cronograma_biorreactor (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    lote_nombre TEXT NOT NULL,
    hora_inicio REAL NOT NULL,
    hora_fin REAL NOT NULL,
    duracion_horas INTEGER NOT NULL,
    fecha_registro TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")
conn.commit()

# 4. INSERTAR DATOS: Limpiamos la tabla anterior para no duplicar datos en las pruebas
cursor.execute("DELETE FROM cronograma_biorreactor")

# Insertamos las filas del cronograma óptimo que calculaste recién
# Usamos una lista de tuplas basada en la salida del optimizador de PuLP
datos_a_guardar = [
    ("Lote_C", 0.0, 3.0, 3),
    ("Lote_B", 3.0, 9.0, 6),
    ("Lote_A", 9.0, 13.0, 4)
]

cursor.executemany("""
INSERT INTO cronograma_biorreactor (lote_nombre, hora_inicio, hora_fin, duracion_horas)
VALUES (?, ?, ?, ?)
""", datos_a_guardar)

conn.commit()
print("¡Base de datos SQL creada y datos insertados con éxito!")

# 5. CONSULTAR DATOS (SELECT): Para comprobar que quedaron bien guardados
print("\n--- CONSULTA DE COMPROBACIÓN DESDE SQLITE ---")
cursor.execute("SELECT lote_nombre, hora_inicio, hora_fin FROM cronograma_biorreactor ORDER BY hora_inicio ASC")
filas = cursor.fetchall()

for fila in filas:
    print(f"Lote en BD: {fila[0]} | Horario: {fila[1]} a {fila[2]} hrs.")

# 6. Cerrar la conexión de manera segura
conn.close()

¡Base de datos SQL creada y datos insertados con éxito!

--- CONSULTA DE COMPROBACIÓN DESDE SQLITE ---
Lote en BD: Lote_C | Horario: 0.0 a 3.0 hrs.
Lote en BD: Lote_B | Horario: 3.0 a 9.0 hrs.
Lote en BD: Lote_A | Horario: 9.0 a 13.0 hrs.
